# 08 — Pull SIPRI Military Data

**Source:** Stockholm International Peace Research Institute (SIPRI)  
**Provider:** SIPRI  
**Access:** Direct Excel download from SIPRI Military Expenditure Database  
**Coverage:** Global, 1949–present (coverage varies by country and series)  
**Frequency:** Annual  

## What this notebook pulls

| Table | SIPRI series | Variables |
|---|---|---|
| **Military expenditure (USD constant)** | SIPRI MILEX | Spending in constant 2022 USD |
| **Military expenditure (% of GDP)** | SIPRI MILEX | Spending as % of GDP |
| **Military expenditure (% of govt spending)** | SIPRI MILEX | Share of government budget |
| **Armed forces personnel** | SIPRI Military Expenditure DB | Total active military personnel |

These feed **Domain 3 — Military/Security** predictors in the meta-plan (coup prediction, ILC models).

## Access note
SIPRI data is freely downloadable from https://www.sipri.org/databases/milex
The Excel files are published annually (typically in April). This notebook targets
the 2024 edition. If the URL has rotated, download the files manually and place them
in a local `sipri_files/` directory, then run from the **Parse SIPRI files** cell.

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# SIPRI publishes Excel files via a stable URL pattern. Update the edition year as needed.
# 2024 edition: https://www.sipri.org/sites/default/files/SIPRI-Milex-data-1949-2023.xlsx
SIPRI_MILEX_URL = (
    "https://www.sipri.org/sites/default/files/"
    "SIPRI-Milex-data-1949-2023.xlsx"
)

# Sheet names in the SIPRI MILEX Excel workbook
SIPRI_SHEETS = {
    "Constant (2022) USD": "milex_usd_const",
    "Share of GDP":         "milex_pct_gdp",
    "Share of Govt. spending": "milex_pct_govt",
    "Per capita":           "milex_per_capita",
}

print(f"Run date    : {RUN_DATE}")
print(f"SIPRI URL   : {SIPRI_MILEX_URL}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Download SIPRI MILEX workbook

In [ ]:
print(f"Downloading SIPRI MILEX workbook...")
resp = requests.get(SIPRI_MILEX_URL, timeout=120)
resp.raise_for_status()
sipri_bytes = resp.content
print(f"Downloaded {len(sipri_bytes) / 1024:.0f} KB")

# Inspect available sheets
xl = pd.ExcelFile(io.BytesIO(sipri_bytes))
print(f"Sheet names: {xl.sheet_names}")

## Parse SIPRI sheets

Each SIPRI sheet is a wide country × year table with metadata rows at the top.
We skip the header rows and melt to a tidy long-format panel.

In [ ]:
def parse_sipri_sheet(workbook_bytes: bytes, sheet_name: str, value_col: str) -> pd.DataFrame | None:
    """
    Parse one SIPRI MILEX sheet.
    SIPRI sheets have ~5 header rows followed by country × year data.
    'xxx' in the data = no data / not applicable.
    """
    try:
        # Read raw without header to detect structure
        df_raw = pd.read_excel(
            io.BytesIO(workbook_bytes), sheet_name=sheet_name,
            header=None, dtype=str,
        )
    except Exception as e:
        print(f"  Could not read sheet '{sheet_name}': {e}")
        return None

    # Find the header row: the row where the first numeric year column appears
    header_row = None
    for i, row in df_raw.iterrows():
        if any(str(v).strip().isdigit() and 1940 <= int(str(v).strip()) <= 2030
               for v in row.values if pd.notna(v) and str(v).strip().isdigit()):
            header_row = i
            break

    if header_row is None:
        print(f"  Could not find header row in '{sheet_name}'")
        return None

    df = pd.read_excel(
        io.BytesIO(workbook_bytes), sheet_name=sheet_name,
        header=header_row, dtype=str,
    )

    # Normalise column names
    df.columns = [str(c).strip() for c in df.columns]
    df.dropna(how="all", inplace=True)

    # First column is country/region name in SIPRI sheets
    country_col = df.columns[0]
    df.rename(columns={country_col: "country_name"}, inplace=True)

    # Year columns are 4-digit integers
    year_cols = [c for c in df.columns if str(c).isdigit() and 1940 <= int(c) <= 2030]

    if not year_cols:
        print(f"  No year columns found in '{sheet_name}'")
        return None

    # Melt to long format
    df_long = df[["country_name"] + year_cols].melt(
        id_vars=["country_name"], var_name="year", value_name=value_col
    )
    df_long["year"] = df_long["year"].astype(int)

    # Replace SIPRI no-data markers with NaN
    df_long[value_col] = (
        df_long[value_col]
        .replace(["xxx", "...", ". .", "n/a", "NA", ""], pd.NA)
    )
    df_long[value_col] = pd.to_numeric(df_long[value_col], errors="coerce")

    # Drop rows where both country and value are null
    df_long.dropna(subset=["country_name"], inplace=True)

    # Drop SIPRI regional subtotals (often contain 'total' or are blank)
    df_long = df_long[~df_long["country_name"].str.lower().str.contains("total", na=False)]

    return df_long

In [ ]:
# Map requested sheet names to what actually exists in the workbook
available_sheets = xl.sheet_names
sipri_frames = {}

for sheet_name, col_name in SIPRI_SHEETS.items():
    # Try exact match first, then partial match
    matched = next(
        (s for s in available_sheets if sheet_name.lower() in s.lower()),
        None,
    )
    if matched:
        print(f"Parsing sheet '{matched}' → column '{col_name}'")
        df_sheet = parse_sipri_sheet(sipri_bytes, matched, col_name)
        if df_sheet is not None:
            sipri_frames[col_name] = df_sheet
            print(f"  {len(df_sheet):,} rows | {df_sheet['year'].min()}–{df_sheet['year'].max()}")
    else:
        print(f"Sheet '{sheet_name}' not found in workbook (available: {available_sheets})")

## Merge sheets into a single SIPRI panel

In [ ]:
if sipri_frames:
    # Use first sheet as base, join the rest
    base_col = list(sipri_frames.keys())[0]
    df_sipri = sipri_frames[base_col].copy()

    for col_name, df_sheet in list(sipri_frames.items())[1:]:
        df_sipri = df_sipri.merge(
            df_sheet[["country_name", "year", col_name]],
            on=["country_name", "year"],
            how="outer",
        )

    print(f"SIPRI merged panel: {df_sipri.shape}")
    print(f"Countries: {df_sipri['country_name'].nunique()} | Years: {df_sipri['year'].min()}–{df_sipri['year'].max()}")
    df_sipri.head(3)
else:
    print("WARNING: no SIPRI sheets successfully parsed")
    df_sipri = pd.DataFrame()

## Write to ADLS

In [ ]:
if not df_sipri.empty:
    write_parquet(df_sipri, f"raw/sipri/{RUN_DATE}/sipri_milex_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("SIPRI pull complete")
print("=" * 55)
if not df_sipri.empty:
    print(f"  Rows      : {len(df_sipri):,} country-years")
    print(f"  Countries : {df_sipri['country_name'].nunique()}")
    print(f"  Years     : {df_sipri['year'].min()}–{df_sipri['year'].max()}")
    print(f"  Columns   : {list(df_sipri.columns)}")
print()
print("ADLS path written:")
print(f"  raw/sipri/{RUN_DATE}/sipri_milex_panel.parquet")